# Stock Market Trend Analysis (SMA & EMA)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Phuvanenthran-P/data-analytics-projects/blob/main/stock-market-analysis/stock_market_analysis.ipynb)

This notebook fetches intraday stock data from Yahoo Finance and analyzes price trends using a 50-period Simple Moving Average (SMA) and Exponential Moving Average (EMA). It includes static and interactive charts, crossover detection, and a timed live refresh.

**Tools:** Python, Pandas, yfinance, Matplotlib, Plotly

## 1. Install and import libraries

In [ ]:
!pip install -q yfinance plotly

In [ ]:
import time
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from IPython.display import clear_output

## 2. Choose a stock
Change `TICKER` to analyze any listed stock (e.g. `MSFT`, `GOOGL`, `TSLA`).

In [ ]:
TICKER = "AAPL"
WINDOW = 50  # moving average period

stock = yf.Ticker(TICKER)

## 3. Collect data
One day of data at 1-minute intervals. Intraday data is only available when markets are open or for the most recent trading day.

In [ ]:
df = stock.history(period="1d", interval="1m")
df.head()

## 4. Clean data and calculate SMA / EMA
- **SMA** averages the last 50 closing prices equally.
- **EMA** gives more weight to recent prices, so it reacts faster.

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

df = df.dropna()

df["SMA"] = df["Close"].rolling(window=WINDOW).mean()
df["EMA"] = df["Close"].ewm(span=WINDOW, adjust=False).mean()

df[["Close", "SMA", "EMA"]].tail()

## 5. Crossover signals
When the EMA moves above the SMA it suggests upward momentum (bullish); below suggests downward momentum (bearish).

In [ ]:
valid = df.dropna(subset=["SMA"]).copy()
valid["Signal"] = (valid["EMA"] > valid["SMA"]).astype(int)
valid["Crossover"] = valid["Signal"].diff()

bullish = valid[valid["Crossover"] == 1]
bearish = valid[valid["Crossover"] == -1]

print(f"Bullish crossovers (EMA crosses above SMA): {len(bullish)}")
print(f"Bearish crossovers (EMA crosses below SMA): {len(bearish)}")

## 6. Static chart (Matplotlib)

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(df["Close"], label="Stock Price", color="blue")
plt.plot(df["SMA"], label=f"{WINDOW}-Period SMA", color="orange")
plt.plot(df["EMA"], label=f"{WINDOW}-Period EMA", color="red")
plt.title(f"{TICKER} Price with SMA and EMA")
plt.xlabel("Time")
plt.ylabel("Price (USD)")
plt.legend(loc="upper left")
plt.show()

## 7. Interactive chart (Plotly)
Hover over points, zoom, and pan to explore.

In [ ]:
def build_figure(data):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=data.index, y=data["Close"], mode="lines", name="Stock Price"))
    fig.add_trace(go.Scatter(x=data.index, y=data["SMA"], mode="lines", name=f"{WINDOW}-Period SMA"))
    fig.add_trace(go.Scatter(x=data.index, y=data["EMA"], mode="lines", name=f"{WINDOW}-Period EMA"))
    fig.update_layout(
        title=f"{TICKER} Price with SMA & EMA",
        xaxis_title="Time",
        yaxis_title="Price (USD)",
        template="plotly_dark",
    )
    return fig

build_figure(df).show()

## 8. Live refresh (limited)
Re-fetches the data every minute for a fixed number of updates, then stops. Increase `REFRESHES` for longer monitoring.

In [ ]:
REFRESHES = 3
SECONDS_BETWEEN = 60

for i in range(REFRESHES):
    live = stock.history(period="1d", interval="1m").dropna()
    live["SMA"] = live["Close"].rolling(window=WINDOW).mean()
    live["EMA"] = live["Close"].ewm(span=WINDOW, adjust=False).mean()

    clear_output(wait=True)
    print(f"Update {i + 1} of {REFRESHES}")
    build_figure(live).show()

    if i < REFRESHES - 1:
        time.sleep(SECONDS_BETWEEN)

## Conclusion
The notebook shows how moving averages smooth intraday price noise and highlight trend direction. The EMA reacts faster than the SMA, and crossovers between them give simple momentum signals.

**Limitations:** Yahoo Finance data can be delayed and rate-limited, and moving averages lag price, so they are not predictive. This is an educational project, not financial advice.

**Next steps:** compare multiple stocks, add RSI and MACD, and deploy as a Streamlit app.